这一章是后端面试里从“八股”进入“项目设计”的部分。

前面 Go 基础、并发、MySQL、Redis、网络这些内容更多是在问你单点技术是否扎实；到了分布式、微服务、高并发、分库分表，面试官真正想看的不是你能不能背出 CAP、BASE、MQ、Redis、分库分表这些词，而是你能不能把它们放到一个真实系统里说明白：

第一，为什么系统要拆。
不是所有项目都需要微服务。单体简单、事务清晰、调用链短。只有当业务复杂度、团队规模、流量压力、发布频率上来之后，微服务才有收益。

第二，拆了以后多了什么问题。
服务调用变远程了，事务不再天然一致，日志不在一台机器上，链路变长，下游会超时，流量会打爆某个节点，数据库会出现瓶颈。

第三，Go 在这里有什么特点。
Go 的优势是 goroutine 轻量、网络 IO 模型简单、context 适合做超时取消和链路透传、标准库 net/http 足够强、gRPC 生态成熟、middleware/interceptor 很适合做日志、鉴权、限流、追踪、熔断等横切逻辑。

第四，什么时候用什么方案。
高并发不是固定套餐，不是看到高并发就 Redis + MQ + 分库分表。真正工程化的回答应该是先看瓶颈，再选方案：读多先缓存，写多看异步削峰，数据库慢看索引和 SQL，热点集中做热点隔离，数据量太大再考虑分库分表。

第五，分库分表不是银弹。
分库分表能解决单库单表容量和并发瓶颈，但会带来跨片查询、分布式事务、分页排序、扩容迁移、全局唯一 ID 等新问题。所以它一定是业务规模逼出来的，而不是项目一开始就默认上。

本章目标是让你形成一套面试回答主线：

> 分布式系统设计的核心不是堆技术名词，而是在一致性、可用性、性能、复杂度之间做取舍。Go 项目中要用 context 管住调用边界，用 middleware/interceptor 做治理，用 goroutine 和 channel 做并发编排，用 Redis/MQ/数据库分片解决具体瓶颈，用指标、日志、链路追踪保证系统可观测。

## 11.1 分布式理论
### 11.1.1 Q：什么是分布式系统？
分布式系统可以理解为：一个完整业务能力不再由单台机器、单个进程、单个数据库完成，而是由多个节点通过网络协作完成。

比如一个电商下单流程，可能涉及：

* 网关服务
* 用户服务
* 商品服务
* 库存服务
* 订单服务
* 支付服务
* 优惠券服务
* 消息服务
* MySQL
* Redis
* MQ

从用户视角看，他只是点了一次“提交订单”。
但从系统内部看，这个请求可能跨了多个服务、多个数据库、多个缓存、多个消息队列。

分布式系统的核心难点来自三个事实：

第一，网络是不可靠的。
调用可能超时、丢包、连接断开、响应慢。

第二，节点是不可靠的。
机器可能宕机，容器可能重启，服务可能发布，进程可能 OOM。

第三，数据副本之间同步需要时间。
只要数据有多个副本，就一定会有一致性和延迟之间的权衡。

所以分布式系统面试的核心不是“怎么调用另一个服务”，而是：

> 在网络、节点、数据都可能不稳定的情况下，如何让系统整体仍然可用、可观测、可恢复。

### 11.1.2 Q：什么是 CAP 理论？
CAP 理论是分布式系统里最基础的理论之一。

CAP 代表三个特性：

C 是 Consistency，一致性。
指的是所有节点在同一时刻看到的数据是一致的。比如我在节点 A 写入数据，马上到节点 B 读取，也能读到最新值。

A 是 Availability，可用性。
指的是系统对每个请求都能给出响应，不会长时间无响应，也不会直接拒绝服务。

P 是 Partition Tolerance，分区容错性。
指的是当网络出现分区，也就是部分节点之间无法通信时，系统仍然能继续工作。

CAP 的核心结论是：

> 在分布式系统中，网络分区不可避免，所以 P 基本必须保留。当 P 发生时，系统只能在 C 和 A 之间做取舍。

如果选择 CP，就是优先一致性，牺牲部分可用性。
比如网络分区时，为了避免读到错误数据，系统可能拒绝写入或拒绝服务。

如果选择 AP，就是优先可用性，牺牲强一致性。
比如网络分区时，系统继续响应请求，但不同节点上的数据可能短时间不一致，后续再通过同步或补偿达到最终一致。

> CAP 不是让系统永久固定选择 CP 或 AP，而是在网络分区发生时，不同业务场景对一致性和可用性的取舍不同。金融、支付、库存这类场景更偏 CP；点赞数、浏览量、推荐统计这类场景可以更偏 AP 或最终一致。

#### 追问：CAP 理论有什么局限性？
CAP 最大的问题是太抽象，也太二元。

现实系统不是简单地“要一致性”或者“要可用性”。

第一，一致性有很多级别。
有强一致、线性一致、顺序一致、读己之写、最终一致。不是所有业务都要求强一致。

第二，可用性也有程度。
系统可以核心功能可用，非核心功能降级；可以部分用户可用，部分用户排队；可以读可用，写受限。

第三，CAP 没有直接讨论延迟。
但现实中延迟非常关键。一个系统理论上最终能响应，但如果每次响应 10 秒，对用户来说也接近不可用。

所以面试里可以接一句 BASE：

> CAP 说明了分布式系统里的根本取舍，BASE 则更偏互联网系统中的工程实践：牺牲强一致，换取基本可用和最终一致。

### 11.1.3 Q：什么是 BASE 理论？
BASE 是对 CAP 的补充，常用于解释互联网系统为什么大量采用最终一致性。

BASE 包括三个部分：

Basically Available，基本可用。
系统出现故障时，不追求所有功能都完美可用，而是保证核心链路可用。比如大促时关闭个性化推荐、降级排行榜，但下单和支付要尽量保住。

Soft State，软状态。
系统允许存在中间状态。比如订单从“待支付”到“已支付”之间可能有“支付中”；库存扣减和订单创建之间可能短暂不一致。

Eventually Consistent，最终一致性。
系统不要求所有节点立刻一致，但保证在一段时间后，通过消息、重试、补偿、对账等机制达到一致。

在 Go 项目里，BASE 的落地经常体现在：

* 用 MQ 异步处理非核心流程
* 用状态机表达中间状态
* 用定时任务做补偿扫描
* 用幂等键保证重试安全
* 用对账任务修复极端异常
* 用 context timeout 控制调用边界
* 用日志和 trace 保证问题可排查

面试里可以这样回答：

> BASE 的核心是承认分布式系统里强一致成本很高，所以对非强一致业务采用最终一致。比如下单成功后，积分、通知、埋点可以通过 MQ 异步完成，只要消息可靠、消费幂等、失败可补偿，就能在保证用户体验的同时提高系统吞吐。

## 11.2 分布式事务
### 11.2.1 Q：为什么会有分布式事务问题？
在单体应用里，如果订单表、库存表、支付表都在同一个数据库里，我们可以用本地事务保证一致性：
```go
tx, err := db.BeginTx(ctx, nil)
```
然后在一个事务里完成扣库存、创建订单、写流水。

但微服务拆分之后，订单服务有自己的库，库存服务有自己的库，支付服务有自己的库。一次业务操作需要跨多个服务和多个数据库，本地事务就不够用了。

比如下单流程：

1. 订单服务创建订单
2. 库存服务扣减库存
3. 优惠券服务核销优惠券
4. 支付服务创建支付单

如果订单创建成功，但库存扣减失败怎么办？
如果库存扣减成功，但订单服务超时了怎么办？
如果支付成功，但通知订单服务失败怎么办？

这就是分布式事务问题。

本质上是：

> 一次业务操作跨越了多个独立资源，而这些资源之间没有天然的原子提交能力。
### 11.2.2 Q：常见分布式事务方案有哪些？
常见方案有四类：

第一，2PC，两阶段提交。
协调者先询问所有参与者能不能提交，所有人都准备好后，再统一提交。它一致性强，但性能差、阻塞严重、协调者压力大，互联网业务中不常直接用。

第二，TCC。
Try、Confirm、Cancel 三阶段。Try 预留资源，Confirm 真正提交，Cancel 回滚资源。它适合对一致性要求高、业务可以显式拆出预留和确认逻辑的场景，比如资金、库存。

第三，Saga。
把长事务拆成多个本地事务，每个本地事务都有对应补偿操作。某一步失败后，按反向顺序执行补偿。适合长流程业务，比如订单履约、旅游预订。

第四，本地消息表 / 事务消息 / Outbox Pattern。
业务数据和消息记录在同一个本地事务里提交，再由后台任务或消息 relay 把消息发出去。这样可以保证“业务写库”和“消息一定会被发出”之间的一致性。

在 Go 项目里，更常见的是：

* 本地事务 + MQ 最终一致
* 消费端幂等
* 失败重试
* 补偿任务
* 对账任务
* 状态机兜底

面试里可以这样回答：

> 强一致分布式事务成本很高。互联网业务里更多采用最终一致方案，比如本地事务写订单和 outbox 消息，再异步投递 MQ，消费者幂等处理，失败重试，最后用补偿任务和对账任务兜底。只有库存冻结、资金划转这类强一致场景，才考虑 TCC 或更严格的事务方案。

---

## 【文本块 8】Go 项目里的事务边界怎么设计？

在 Go 里，事务边界不要散落在很多 repository 里。

比较推荐的结构是：

* handler 只负责参数解析和响应
* application service 负责编排业务流程和事务边界
* domain 层负责业务规则
* repository 负责数据访问
* infra 层负责 DB、Redis、MQ、RPC 等外部依赖

也就是说，事务一般放在应用服务层，而不是每个 DAO 自己开事务。

错误风格：

```text
OrderRepo.Create 内部自己开事务
InventoryRepo.Deduct 内部自己开事务
CouponRepo.Use 内部自己开事务
```

这样上层无法保证整体一致性。

更好的方式是：

```text
Application Service 开启 tx
OrderRepo 使用 tx 写订单
OutboxRepo 使用 tx 写消息
提交 tx
后台 relay 发送消息
```

---

## 【代码块 1】本地事务 + Outbox 思路示例

```go
package main

import (
	"context"
	"database/sql"
	"encoding/json"
	"fmt"
)

type Order struct {
	ID     int64
	UserID int64
	Amount int64
}

type OutboxMessage struct {
	Topic string
	Key   string
	Body  []byte
}

func CreateOrderWithOutbox(ctx context.Context, db *sql.DB, order Order) error {
	tx, err := db.BeginTx(ctx, nil)
	if err != nil {
		return fmt.Errorf("begin tx: %w", err)
	}
	defer tx.Rollback()

	_, err = tx.ExecContext(
		ctx,
		"INSERT INTO orders(id, user_id, amount, status) VALUES (?, ?, ?, ?)",
		order.ID,
		order.UserID,
		order.Amount,
		"CREATED",
	)
	if err != nil {
		return fmt.Errorf("insert order: %w", err)
	}

	body, err := json.Marshal(order)
	if err != nil {
		return fmt.Errorf("marshal order event: %w", err)
	}

	_, err = tx.ExecContext(
		ctx,
		"INSERT INTO outbox(topic, msg_key, body, status) VALUES (?, ?, ?, ?)",
		"order.created",
		fmt.Sprintf("%d", order.ID),
		body,
		"NEW",
	)
	if err != nil {
		return fmt.Errorf("insert outbox: %w", err)
	}

	if err := tx.Commit(); err != nil {
		return fmt.Errorf("commit tx: %w", err)
	}

	return nil
}
```

---

## 【文本块 9】代码解释

这段代码体现的是 Outbox Pattern 的核心思想：

订单数据和待发送消息在同一个本地事务里提交。

只要订单创建成功，outbox 表里一定有一条待发送消息。
后续可以由后台 goroutine、定时任务或独立 relay 服务扫描 outbox 表，把消息投递到 Kafka、RocketMQ 或其他消息系统。

这样避免了一个经典问题：

> 订单写库成功，但发送 MQ 失败，导致下游永远不知道订单创建了。

当然，Outbox 不是结束，它还需要配合：

* outbox 表状态流转
* 投递失败重试
* 消息 key 去重
* 消费端幂等
* 死信队列或人工补偿
* 定期对账

---

# 三、幂等性

## 【文本块 10】Q：为什么分布式系统里一定要考虑幂等？

因为分布式系统里重试非常常见。

服务调用可能超时，调用方不知道对方到底有没有执行成功。
MQ 消息可能重复投递。
用户可能重复点击。
网关可能重试。
定时任务可能重复扫描。
消费者可能处理成功但提交 offset 失败。

如果接口不幂等，重试就可能导致严重问题：

* 重复下单
* 重复扣款
* 重复发券
* 重复扣库存
* 重复发送通知
* 重复创建记录

所以分布式系统里有一句很重要的话：

> 只要你允许重试，就必须考虑幂等。

---

## 【文本块 11】Q：常见幂等方案有哪些？

常见幂等方案有几类：

第一，唯一索引。
比如订单号、支付流水号、请求 ID 建唯一索引。重复请求插入时数据库直接拒绝。

第二，幂等表。
把业务唯一请求号记录下来，处理前先查，处理成功后记录状态。

第三，Redis SETNX。
用请求 ID 作为 key，先抢占执行权，成功才继续处理。

第四，状态机校验。
比如订单只能从 CREATED 变成 PAID，不能从 PAID 再变成 PAID 并重复扣款。

第五，乐观锁。
通过 version 字段保证更新只发生一次。

第六，MQ 消费去重。
消费端根据 message_id 或业务 key 做去重，保证重复消息不会重复产生业务影响。

面试里可以这样回答：

> 我一般不会只依赖一种幂等手段。对于核心写接口，会用业务唯一键或幂等号加唯一索引兜底；对于状态流转，用状态机限制非法重复更新；对于 MQ 消费，用消息 ID 或业务 key 去重，保证重复投递不会重复执行业务。

---

## 【代码块 2】数据库唯一索引实现幂等的思路

```go
package main

import (
	"context"
	"database/sql"
	"fmt"
)

func CreatePayment(ctx context.Context, db *sql.DB, requestID string, orderID int64, amount int64) error {
	_, err := db.ExecContext(
		ctx,
		"INSERT INTO payments(request_id, order_id, amount, status) VALUES (?, ?, ?, ?)",
		requestID,
		orderID,
		amount,
		"CREATED",
	)
	if err != nil {
		// 实际项目中要根据数据库驱动判断是否是 duplicate key。
		// 如果是重复请求，可以查询原支付单并返回成功或当前状态。
		return fmt.Errorf("insert payment: %w", err)
	}
	return nil
}
```

---

## 【文本块 12】追问：接口超时了，客户端该不该重试？

可以重试，但前提是接口幂等。

比如支付创建接口，客户端带一个 request_id。
服务端以 request_id 作为唯一键。
第一次请求如果服务端已经处理成功，但响应在网络中丢了，客户端第二次重试时，服务端发现 request_id 已存在，就返回第一次的处理结果，而不是重新创建一笔支付。

这就是幂等号的价值。

如果没有幂等号，重试就是危险的。

---

# 四、分布式锁

## 【文本块 13】Q：什么是分布式锁？什么时候需要？

分布式锁用于在多个进程、多个机器之间控制同一份资源的并发访问。

单进程内，我们可以用 Go 的 sync.Mutex：

```go
var mu sync.Mutex
```

但如果你的服务部署了 10 个实例，每个实例都有自己的内存锁，那么 sync.Mutex 只能保护本进程内的并发，不能保护多个实例之间的并发。

这时就需要分布式锁。

常见场景：

* 防止多个实例同时执行同一个定时任务
* 防止同一用户重复下单
* 防止热点资源被并发修改
* 防止缓存重建时大量请求同时打到数据库
* 控制全局唯一任务的执行权

常见实现：

* Redis 分布式锁
* etcd 分布式锁
* ZooKeeper 分布式锁
* 数据库唯一约束或悲观锁

Go 项目里最常见的是 Redis 或 etcd。

---

## 【文本块 14】Q：Redis 分布式锁需要注意什么？

最基本的 Redis 锁是：

```text
SET lock_key unique_value NX PX 30000
```

含义是：

* NX：只有 key 不存在时才设置成功
* PX：设置过期时间，避免死锁
* unique_value：锁持有者标识，用于安全释放

释放锁时，不能直接 DEL。

因为可能出现：

1. 线程 A 拿到锁，执行业务很久
2. 锁过期自动释放
3. 线程 B 拿到锁
4. 线程 A 执行完，直接 DEL，把 B 的锁删了

所以释放锁必须判断 value 是不是自己的，并且判断和删除要用 Lua 保证原子性。

面试里可以这样回答：

> Redis 分布式锁至少要满足：加锁用 SET NX PX，锁 value 用唯一标识，释放锁用 Lua 判断 value 后删除，业务时间可能超过锁过期时间时要考虑续期。更严肃的场景要评估 Redis 主从切换、Redlock 或 etcd 锁。

---

## 【代码块 3】Redis 锁伪代码结构

```go
package main

import (
	"context"
	"crypto/rand"
	"encoding/hex"
	"fmt"
	"time"
)

type RedisClient interface {
	SetNX(ctx context.Context, key string, value any, expiration time.Duration) (bool, error)
	Eval(ctx context.Context, script string, keys []string, args ...any) (any, error)
}

func randomToken() string {
	b := make([]byte, 16)
	_, _ = rand.Read(b)
	return hex.EncodeToString(b)
}

type RedisLock struct {
	client RedisClient
	key    string
	token  string
	ttl    time.Duration
}

func NewRedisLock(client RedisClient, key string, ttl time.Duration) *RedisLock {
	return &RedisLock{
		client: client,
		key:    key,
		token:  randomToken(),
		ttl:    ttl,
	}
}

func (l *RedisLock) Lock(ctx context.Context) (bool, error) {
	ok, err := l.client.SetNX(ctx, l.key, l.token, l.ttl)
	if err != nil {
		return false, fmt.Errorf("redis setnx lock: %w", err)
	}
	return ok, nil
}

func (l *RedisLock) Unlock(ctx context.Context) error {
	const script = `
if redis.call("GET", KEYS[1]) == ARGV[1] then
    return redis.call("DEL", KEYS[1])
else
    return 0
end
`
	_, err := l.client.Eval(ctx, script, []string{l.key}, l.token)
	if err != nil {
		return fmt.Errorf("redis unlock: %w", err)
	}
	return nil
}
```

---

## 【文本块 15】追问：分布式锁是不是万能的？

不是。

分布式锁会引入额外复杂度：

* 锁超时时间不好设置
* 业务执行时间可能超过锁 TTL
* 续期机制复杂
* Redis 主从切换可能带来一致性问题
* 锁粒度太大影响吞吐
* 锁失败后业务怎么处理也要设计

很多场景其实不一定要分布式锁。

比如防重复下单，可以用唯一索引和幂等号。
比如库存扣减，可以用数据库条件更新或 Redis 原子扣减。
比如任务只执行一次，可以用数据库任务表抢占。

面试里最好补一句：

> 我不会一遇到并发就上分布式锁。能用唯一索引、状态机、乐观锁解决的，优先用数据层约束。分布式锁更适合确实需要跨实例互斥执行的场景。

---

# 五、分布式 ID

## 【文本块 16】Q：为什么需要分布式 ID？

在单库单表里，可以直接用数据库自增 ID。

但在分布式系统里，多个服务、多个数据库、多个分片都可能同时生成 ID。此时如果还依赖单库自增，就会有几个问题：

第一，单点瓶颈。
所有 ID 都从一个数据库生成，压力集中。

第二，不利于分库分表。
多个分片自增 ID 可能冲突。

第三，扩展性差。
服务和库越多，越需要一个全局唯一 ID 方案。

一个好的分布式 ID 通常希望满足：

* 全局唯一
* 趋势递增
* 高性能
* 高可用
* 信息不要过度暴露
* 长度不要太夸张
* 方便数据库索引

常见方案：

* UUID
* 数据库号段
* Redis incr
* Snowflake 雪花算法
* Leaf、UidGenerator 这类 ID 服务

Go 项目里最常见的是 Snowflake 思路或者独立 ID 服务。

---

## 【文本块 17】Q：Snowflake 雪花算法是什么？

Snowflake 是一种生成 64 位整数 ID 的方案。

经典结构是：

* 1 位符号位
* 41 位时间戳
* 10 位机器 ID
* 12 位序列号

它的特点是：

* 本地生成，不依赖远程服务
* 性能高
* 大体按时间递增
* 适合作为数据库主键
* 可以支持多机器并发生成

但也有问题：

* 依赖机器时钟
* 时钟回拨会导致 ID 重复风险
* 机器 ID 分配要保证不冲突
* 时间戳暴露业务时间信息
* 如果单毫秒内序列号用完，需要等待下一毫秒

面试里可以这样回答：

> Snowflake 通过时间戳、机器 ID、序列号组合生成全局唯一且趋势递增的 64 位 ID。它性能很好，适合高并发写入场景。但要处理机器 ID 分配和时钟回拨问题，不能只背结构不讲风险。

---

## 【代码块 4】简化版 Snowflake ID 生成器

```go
package main

import (
	"fmt"
	"sync"
	"time"
)

type Snowflake struct {
	mu            sync.Mutex
	epoch         int64
	workerID      int64
	sequence      int64
	lastTimestamp int64
}

const (
	workerIDBits  = 10
	sequenceBits  = 12
	maxSequence   = int64(-1) ^ (int64(-1) << sequenceBits)
	workerIDShift = sequenceBits
	timeShift     = sequenceBits + workerIDBits
)

func NewSnowflake(workerID int64) *Snowflake {
	return &Snowflake{
		epoch:    time.Date(2024, 1, 1, 0, 0, 0, 0, time.UTC).UnixMilli(),
		workerID: workerID,
	}
}

func currentMillis() int64 {
	return time.Now().UnixMilli()
}

func (s *Snowflake) NextID() (int64, error) {
	s.mu.Lock()
	defer s.mu.Unlock()

	now := currentMillis()
	if now < s.lastTimestamp {
		return 0, fmt.Errorf("clock moved backwards: now=%d last=%d", now, s.lastTimestamp)
	}

	if now == s.lastTimestamp {
		s.sequence = (s.sequence + 1) & maxSequence
		if s.sequence == 0 {
			for now <= s.lastTimestamp {
				now = currentMillis()
			}
		}
	} else {
		s.sequence = 0
	}

	s.lastTimestamp = now

	id := ((now - s.epoch) << timeShift) |
		(s.workerID << workerIDShift) |
		s.sequence

	return id, nil
}

func main() {
	sf := NewSnowflake(1)

	for i := 0; i < 5; i++ {
		id, err := sf.NextID()
		if err != nil {
			panic(err)
		}
		fmt.Println(id)
	}
}
```

---

## 【文本块 18】代码解释

这个实现只是教学版，能帮助你理解 Snowflake 的核心结构。

真实生产里还要补充：

* workerID 如何分配
* workerID 是否会重复
* 时钟回拨怎么处理
* 多实例重启时如何恢复
* 是否需要把机房 ID、业务线 ID 编码进去
* 是否允许暴露时间趋势
* 是否需要压缩成字符串

面试里不要只说“我们用雪花算法”，要能说出它的风险和治理方式。

---

# 六、微服务架构

## 【文本块 19】Q：什么是微服务？

微服务可以理解为：把一个大系统按业务能力拆成多个小服务，每个服务可以独立开发、独立部署、独立扩容、独立演进。

它的重点不是“服务越小越好”，而是：

* 业务边界清晰
* 职责单一
* 数据尽量自治
* 可以独立发布
* 可以独立扩容
* 团队可以独立负责

比如电商系统可以拆成：

* 用户服务
* 商品服务
* 库存服务
* 订单服务
* 支付服务
* 营销服务
* 搜索服务
* 通知服务

面试里可以这样回答：

> 微服务本质上是按业务能力拆分系统，用分布式协作替代单体内部调用。它解决的是团队协作、独立发布、独立扩容、复杂度隔离的问题，但也引入了远程调用、分布式事务、服务治理和可观测性成本。

---

## 【文本块 20】Q：什么时候需要从单体演进到微服务？

不是所有系统都应该一开始就微服务。

单体有很多优点：

* 开发简单
* 本地调用快
* 事务好处理
* 部署链路短
* 排查问题容易
* 早期迭代效率高

当出现下面几类问题时，才考虑拆微服务：

第一，团队规模变大。
很多人同时改一个单体，代码冲突、发布协调、模块边界都变得困难。

第二，业务复杂度上升。
订单、支付、库存、营销、履约等模块相互影响，改一个地方容易牵一发而动全身。

第三，流量差异明显。
比如商品详情流量很高，后台管理流量很低。如果都在一个单体里，就很难单独扩容热点模块。

第四，发布频率不同。
营销模块频繁改，支付模块要稳定。如果放在同一个单体里，每次营销发布都会影响支付风险。

第五，故障隔离需要。
一个非核心模块 bug 不应该拖垮整个核心交易链路。

面试里可以这样答：

> 我不会为了微服务而微服务。早期业务不复杂时，单体更快。只有当团队协作、模块耦合、独立扩容、独立发布、故障隔离这些问题开始明显影响效率和稳定性时，才逐步拆服务。

---

## 【文本块 21】Q：微服务怎么拆？

最推荐的是按业务边界拆，而不是按技术层拆。

推荐拆法：

* 按业务领域拆：用户、订单、库存、支付
* 按核心链路拆：交易、营销、履约
* 按通用能力拆：搜索、推荐、通知、风控

不推荐拆法：

* Controller 服务
* Service 服务
* DAO 服务
* Redis 服务
* MySQL 服务

这种按技术层拆的方式没有业务闭环，只会把本来单体内部的一次方法调用变成跨网络调用，徒增复杂度。

判断服务粒度是否合适，可以看三个问题：

第一，这个服务有没有清晰业务边界？
第二，高频调用是否大多发生在服务内部？
第三，它是否真的有独立发布和独立扩容价值？

面试里可以说：

> 服务不是越小越好。太大会退化成小单体，太小会导致调用链过长、事务复杂、排查困难。比较合理的粒度是围绕业务能力闭环，让一个服务内部能完成大部分高频逻辑，对外暴露稳定 API。

---

## 【文本块 22】Go 微服务一般怎么组织代码？

Go 没有像 Java Spring 那样强绑定的项目结构，但工程里通常会形成类似分层：

```text
cmd/
  order-service/
    main.go

internal/
  app/
    order/
      service.go
      handler.go
      repository.go
      model.go
  pkg/
    middleware/
    config/
    logger/
    tracing/
    client/

api/
  proto/
  openapi/

configs/
```

常见分层：

handler 层：处理 HTTP/gRPC 请求，做参数解析、响应转换。
application service 层：编排业务流程、事务、调用外部依赖。
domain 层：沉淀核心业务规则。
repository 层：封装数据库访问。
infra 层：封装 Redis、MQ、RPC client、配置、日志等基础设施。

Go 里不要机械模仿 Java 的 Controller、Service、DAO 三层。
更重要的是：

* 依赖方向清楚
* 业务逻辑不要散在 handler 里
* 数据访问不要污染领域逻辑
* 外部依赖通过 interface 隔离
* context 从入口向下传递
* error 在边界层统一转换

---

# 七、Go 微服务通信

## 【文本块 23】Q：微服务之间常见通信方式有哪些？

主要分两类：

第一，同步调用。
常见是 HTTP 或 gRPC。调用方需要立刻拿到结果。

适合：

* 查询用户信息
* 校验库存
* 创建支付单
* 获取订单详情

第二，异步通信。
常见是 MQ 或事件驱动。调用方不需要立刻等待下游完成。

适合：

* 发送通知
* 发放积分
* 写埋点
* 异步扣减
* 异步生成报表
* 削峰填谷

Go 项目里，同步调用常用：

* net/http
* gRPC
* Hertz
* Gin
* Echo
* Fiber
* Kitex
* Kratos
* go-zero

异步通信常用：

* Kafka client
* RocketMQ client
* RabbitMQ client
* Redis Stream
* NATS

面试里可以这样回答：

> 需要立即返回结果的链路用同步调用，比如下单前校验库存；非核心、可异步、可最终一致的流程用 MQ，比如通知、积分、埋点。真实系统通常是同步和异步混合，而不是二选一。

---

## 【文本块 24】Q：Go 服务调用为什么一定要重视 context？

context 是 Go 微服务里非常核心的调用边界控制工具。

它主要解决三类问题：

第一，超时控制。
一次请求不能无限等下游。入口设置 timeout，下游 DB、Redis、RPC、HTTP 都应该使用这个 ctx。

第二，取消传播。
如果用户断开连接、请求超时、上游取消，下游 goroutine 和远程调用应该尽快停止，避免资源浪费。

第三，链路信息传递。
比如 trace_id、request_id、user_id、灰度标识等，可以通过 context 或 header 透传。

Go 项目中很忌讳：

```go
context.Background()
```

在业务链路中到处乱用。

因为这样会切断超时、取消和 trace。

更好的习惯是：

```go
func (s *Service) CreateOrder(ctx context.Context, req CreateOrderReq) error
```

所有下游调用都继续使用这个 ctx。

---

## 【代码块 5】Go HTTP 服务中传递 context

```go
package main

import (
	"context"
	"fmt"
	"net/http"
	"time"
)

type OrderService struct{}

func (s *OrderService) CreateOrder(ctx context.Context, userID string) error {
	select {
	case <-time.After(50 * time.Millisecond):
		fmt.Println("order created for user:", userID)
		return nil
	case <-ctx.Done():
		return ctx.Err()
	}
}

func main() {
	svc := &OrderService{}

	http.HandleFunc("/orders", func(w http.ResponseWriter, r *http.Request) {
		ctx, cancel := context.WithTimeout(r.Context(), 200*time.Millisecond)
		defer cancel()

		if err := svc.CreateOrder(ctx, "u1001"); err != nil {
			http.Error(w, err.Error(), http.StatusGatewayTimeout)
			return
		}

		w.WriteHeader(http.StatusOK)
		_, _ = w.Write([]byte("ok"))
	})

	_ = http.ListenAndServe(":8080", nil)
}
```

---

## 【文本块 25】代码解释

这段代码里，handler 从 `r.Context()` 派生出带超时的 ctx。

后续 service 层不自己创建 Background，而是继续使用入口传下来的 ctx。

这样有几个好处：

* 请求超时后，下游逻辑可以及时停止
* 客户端断开后，服务端能感知取消
* 后续如果接入 trace，也可以沿着 ctx 传递
* 服务间调用能形成统一的 deadline

面试里可以说：

> Go 微服务里 context 是请求生命周期的主线。只要进入一个请求，就应该把 ctx 一路传到 DB、Redis、RPC、MQ 相关操作里，不要在中间用 Background 断链。

---

# 八、服务治理与高可用

## 【文本块 26】Q：什么是服务治理？

服务治理就是微服务拆开之后，为了让服务之间稳定协作所做的一整套控制能力。

它包括：

* 服务注册与发现
* 负载均衡
* 超时控制
* 重试
* 熔断
* 限流
* 降级
* 隔离
* 配置中心
* 灰度发布
* 健康检查
* 链路追踪
* 指标监控
* 日志聚合

如果没有服务治理，微服务只是一堆互相 HTTP 调用的进程。
一旦某个下游慢了、挂了、返回异常了，上游可能被拖死，最后雪崩。

面试里可以这样回答：

> 微服务真正难的不是“能调用”，而是“下游慢、节点挂、流量突增、网络抖动时还能稳住”。服务治理就是通过注册发现、负载均衡、超时、重试、熔断、限流、降级、监控追踪等手段保证系统在不稳定环境下仍然可控。

---

## 【文本块 27】Q：超时、重试、熔断、降级分别解决什么问题？

超时解决的是“不无限等待”。
任何远程调用都必须设置超时，否则一个慢下游可能耗尽上游 goroutine、连接池、线程资源。

重试解决的是“偶发失败”。
比如网络抖动、短暂 502、临时超时，可以通过有限重试提高成功率。

熔断解决的是“失败扩散”。
当某个下游持续失败时，上游不要继续大量请求它，而是短时间快速失败，避免把下游和自己一起打死。

降级解决的是“保核心功能”。
当系统压力过大或下游不可用时，关闭非核心功能，保证核心链路可用。

这四者经常一起出现，但不能乱用。

尤其是重试：

> 重试必须建立在幂等前提上，而且要有次数限制、超时限制、退避策略，否则重试本身会放大流量，造成雪崩。

---

## 【文本块 28】Go 项目里服务治理一般放在哪里？

常见位置：

HTTP middleware：
做 request_id、日志、recover、鉴权、限流、trace。

gRPC interceptor：
做 metadata 透传、日志、监控、超时、熔断、错误码转换。

RPC client wrapper：
统一封装下游调用的 timeout、retry、circuit breaker。

Gateway：
做入口鉴权、限流、路由、灰度、协议转换。

Service 层：
做业务降级、兜底数据、幂等控制。

在 Go 里，middleware/interceptor 是非常重要的工程扩展点。

---

## 【代码块 6】HTTP Middleware：日志 + recover + 耗时统计

```go
package main

import (
	"log"
	"net/http"
	"time"
)

func RecoverAndLog(next http.Handler) http.Handler {
	return http.HandlerFunc(func(w http.ResponseWriter, r *http.Request) {
		start := time.Now()

		defer func() {
			cost := time.Since(start)
			log.Printf("method=%s path=%s cost=%s", r.Method, r.URL.Path, cost)

			if rec := recover(); rec != nil {
				log.Printf("panic: %v", rec)
				http.Error(w, "internal server error", http.StatusInternalServerError)
			}
		}()

		next.ServeHTTP(w, r)
	})
}

func main() {
	mux := http.NewServeMux()

	mux.HandleFunc("/hello", func(w http.ResponseWriter, r *http.Request) {
		_, _ = w.Write([]byte("hello"))
	})

	handler := RecoverAndLog(mux)

	_ = http.ListenAndServe(":8080", handler)
}
```

---

## 【文本块 29】代码解释

这个 middleware 做了三件事：

第一，记录请求耗时。
第二，捕获 handler panic，避免单个请求打崩整个进程。
第三，统一返回 500。

真实项目里还会补充：

* trace_id
* request_id
* 用户 ID
* 状态码
* 请求体大小
* 响应体大小
* panic stack
* Prometheus 指标
* OpenTelemetry span

---

# 九、可观测性：日志、指标、链路追踪

## 【文本块 30】Q：微服务场景下为什么需要可观测性？

单体系统排查问题时，可能看一台机器、一份日志就够了。

微服务系统里，一个请求可能经过：

```text
gateway -> order-service -> inventory-service -> payment-service -> coupon-service -> mq -> notify-service
```

如果没有 trace_id，你很难知道这几段日志是不是同一个请求。
如果没有指标，你不知道哪个服务 QPS 涨了、RT 高了、错误率上来了。
如果没有链路追踪，你不知道慢在哪一跳。

所以可观测性通常包括三件事：

日志 Logs：
记录具体事件，方便排查某一次请求发生了什么。

指标 Metrics：
记录系统整体状态，比如 QPS、RT、错误率、CPU、内存、连接池、队列堆积。

链路 Traces：
记录一次请求跨服务的完整调用链，知道每一段耗时和状态。

面试里可以说：

> 微服务排障不能只靠 grep 日志。要把日志、指标、链路追踪打通。日志回答“发生了什么”，指标回答“系统现在怎么样”，trace 回答“慢在哪一段”。

---

## 【文本块 31】Q：Go 微服务里 trace_id 怎么透传？

常见做法：

1. 网关入口生成或接收 trace_id
2. 放入 HTTP Header 或 gRPC metadata
3. 进入服务后写入 context
4. 日志打印 trace_id
5. 调用下游时继续放入 header/metadata
6. MQ 消息发送时写入 message header
7. 消费端从 message header 恢复 trace context

Go 里可以通过 context 实现链路内传递，但跨网络必须通过协议载体：

* HTTP Header
* gRPC metadata
* MQ Header
* Kafka message headers

如果只放在 context 里，不写到 header，下游服务是拿不到的。

---

## 【代码块 7】简化版 trace_id middleware

```go
package main

import (
	"context"
	"crypto/rand"
	"encoding/hex"
	"log"
	"net/http"
)

type traceIDKey struct{}

func newTraceID() string {
	b := make([]byte, 16)
	_, _ = rand.Read(b)
	return hex.EncodeToString(b)
}

func TraceMiddleware(next http.Handler) http.Handler {
	return http.HandlerFunc(func(w http.ResponseWriter, r *http.Request) {
		traceID := r.Header.Get("X-Trace-ID")
		if traceID == "" {
			traceID = newTraceID()
		}

		ctx := context.WithValue(r.Context(), traceIDKey{}, traceID)
		w.Header().Set("X-Trace-ID", traceID)

		next.ServeHTTP(w, r.WithContext(ctx))
	})
}

func TraceIDFromContext(ctx context.Context) string {
	v, _ := ctx.Value(traceIDKey{}).(string)
	return v
}

func main() {
	mux := http.NewServeMux()

	mux.HandleFunc("/hello", func(w http.ResponseWriter, r *http.Request) {
		traceID := TraceIDFromContext(r.Context())
		log.Println("trace_id=", traceID)
		_, _ = w.Write([]byte("hello"))
	})

	_ = http.ListenAndServe(":8080", TraceMiddleware(mux))
}
```

---

## 【文本块 32】追问：context.WithValue 能不能乱用？

不能。

context.WithValue 适合传递请求级元信息，比如 trace_id、request_id、auth info。

不适合传递：

* 普通函数参数
* 可选配置
* 大对象
* 数据库连接
* 业务实体
* 隐式依赖

否则代码会变得非常难维护。

面试里可以说：

> context 主要用于超时、取消和请求级元信息透传，不应该把它当成全局 map。业务参数还是应该显式传参。

---

# 十、高并发系统设计

## 【文本块 33】Q：拿到高并发设计题，第一步看什么？

不要一上来就说 Redis、MQ、分库分表。

第一步应该先问业务约束。

关键问题包括：

* 是读多还是写多？
* 峰值 QPS 大概多少？
* 平均 QPS 和峰值差多少？
* 是否允许异步？
* 一致性要求多高？
* 是否有热点 key？
* 是否有热点用户或热点商品？
* 数据量多大？
* 单条请求耗时主要在哪里？
* 瓶颈是 CPU、内存、IO、数据库还是下游依赖？
* 失败后能否重试？
* 重试是否幂等？
* 核心链路和非核心链路是什么？

面试里最加分的回答是：

> 先抽象业务约束，再谈架构方案。不同业务的高并发瓶颈完全不同，不能一上来套 Redis + MQ + 分库分表。

---

## 【文本块 34】Q：高并发系统常见优化手段有哪些？

可以按层次回答。

第一，流量层：

* CDN
* 静态化
* 网关限流
* 黑白名单
* 验证码
* 排队
* 熔断降级

第二，接入层：

* 无状态服务
* 多实例水平扩容
* 负载均衡
* keep-alive
* 超时控制
* 连接池

第三，应用层：

* 减少远程调用
* 批量化
* 并行化
* 池化
* 本地缓存
* singleflight 请求合并
* goroutine worker pool
* 异步化非核心流程

第四，缓存层：

* Redis 缓存
* 本地缓存
* 多级缓存
* 热点 key 拆分
* 缓存预热
* 缓存穿透、击穿、雪崩治理

第五，存储层：

* SQL 优化
* 索引优化
* 读写分离
* 分库分表
* 冷热数据分离
* 异步写入
* 宽表或冗余表

第六，异步层：

* MQ 削峰填谷
* 事件驱动
* 延迟队列
* 重试队列
* 死信队列
* 补偿任务

第七，稳定性层：

* 限流
* 熔断
* 降级
* 隔离
* 超时
* 重试
* 监控告警
* 链路追踪

面试里要强调：

> 这些手段不是固定套餐，而是按照瓶颈选择。比如读多先看缓存，写峰值高先看削峰，数据库容量顶不住再看分库分表，下游慢先看超时熔断隔离。

---

# 十一、限流

## 【文本块 35】Q：常见限流算法有哪些？

常见限流算法有四种。

第一，固定窗口。
每个时间窗口内允许固定请求数。实现简单，但窗口边界可能出现瞬时双倍流量。

第二，滑动窗口。
把大窗口切成多个小窗口，统计更平滑，但实现稍复杂。

第三，漏桶算法。
请求像水一样进入桶，以固定速率流出。它能平滑流量，但对突发流量不够友好。

第四，令牌桶算法。
系统按固定速率生成令牌，请求拿到令牌才能通过。桶里可以积累一定令牌，所以允许一定突发流量。

工程里常用令牌桶，因为它既能限制平均速率，也能允许合理突发。

限流位置可以在：

* 网关
* 服务入口 middleware
* 用户维度
* IP 维度
* API 维度
* 商户维度
* 下游依赖维度

---

## 【代码块 8】Go 简化版令牌桶限流器

```go
package main

import (
	"fmt"
	"sync"
	"time"
)

type TokenBucket struct {
	mu       sync.Mutex
	rate     int
	capacity int
	tokens   int
	last     time.Time
}

func NewTokenBucket(rate int, capacity int) *TokenBucket {
	return &TokenBucket{
		rate:     rate,
		capacity: capacity,
		tokens:   capacity,
		last:     time.Now(),
	}
}

func (b *TokenBucket) Allow() bool {
	b.mu.Lock()
	defer b.mu.Unlock()

	now := time.Now()
	elapsed := now.Sub(b.last)

	newTokens := int(elapsed.Seconds() * float64(b.rate))
	if newTokens > 0 {
		b.tokens += newTokens
		if b.tokens > b.capacity {
			b.tokens = b.capacity
		}
		b.last = now
	}

	if b.tokens <= 0 {
		return false
	}

	b.tokens--
	return true
}

func main() {
	limiter := NewTokenBucket(2, 5)

	for i := 0; i < 10; i++ {
		fmt.Println(i, limiter.Allow())
		time.Sleep(200 * time.Millisecond)
	}
}
```

---

## 【文本块 36】代码解释

这个简化版令牌桶有两个关键参数：

rate：每秒生成多少令牌。
capacity：桶最多存多少令牌。

当请求来了，先根据距离上次请求的时间补充令牌。
如果有令牌，请求通过并消耗一个。
如果没有令牌，请求被拒绝。

真实生产中还要考虑：

* 分布式限流
* 多维度限流
* Redis Lua 原子限流
* 热点 key
* 限流后的返回码
* 用户体验
* 白名单
* 动态配置
* 监控告警

Go 项目里，单实例限流可以用内存令牌桶；多实例全局限流通常要借助 Redis、网关或专门限流系统。

---

# 十二、削峰填谷与 MQ

## 【文本块 37】Q：什么是削峰填谷？

削峰填谷就是把瞬时高峰请求先缓冲起来，再按照系统能承受的速度慢慢处理。

比如秒杀开始时，1 秒钟来了 100 万请求。
如果这些请求全部直接打到订单服务和数据库，数据库会被打爆。
如果先通过 Redis 预扣库存，再把成功请求写入 MQ，订单服务按自己的消费能力慢慢创建订单，就能把峰值压力摊平。

MQ 的价值主要有：

* 削峰填谷
* 异步解耦
* 失败重试
* 流量缓冲
* 最终一致
* 下游故障隔离

但 MQ 不是免费午餐，它会带来：

* 消息丢失风险
* 消息重复
* 消息乱序
* 消息堆积
* 消费失败
* 最终一致
* 排查复杂度上升

所以面试里可以说：

> MQ 适合非核心同步链路、可异步、可最终一致的场景。用 MQ 以后必须配套解决可靠投递、消费幂等、失败重试、死信队列和消息堆积监控。

---

## 【文本块 38】Q：异步化是不是万能解法？

不是。

异步化可以提高吞吐，降低主链路耗时，但会引入新的问题：

第一，用户感知变复杂。
同步能立即知道成功失败；异步可能只能告诉用户“处理中”。

第二，一致性变复杂。
异步意味着不同系统之间会短暂不一致，需要状态机和补偿机制。

第三，排查变复杂。
请求已经返回了，但后续消费失败，需要 trace 和消息日志才能排查。

第四，重试变复杂。
失败重试可能导致重复消费，所以必须幂等。

第五，顺序性变复杂。
多个消息如果乱序消费，可能导致状态回退。

面试里可以这样回答：

> 异步化适合削峰和解耦，但不是万能解法。它把同步等待的问题变成了最终一致、幂等、补偿和可观测性问题，所以核心链路能不能异步，取决于业务是否接受中间状态。

---

# 十三、热点问题

## 【文本块 39】Q：高并发系统里的热点问题怎么处理？

热点问题通常表现为：

* 某个商品被疯狂访问
* 某个 Redis key 被打爆
* 某个用户或商户流量异常高
* 某张表或某一行被频繁更新
* 某个服务实例压力特别大
* 某个缓存节点成为瓶颈

常见处理方式：

第一，多级缓存。
本地缓存 + Redis + DB，热点数据尽量在更靠近应用的位置命中。

第二，热点 key 拆分。
把一个 key 拆成多个 key，例如 `stock:sku:1:bucket:0~99`，分散访问压力。

第三，请求合并。
同一时刻大量请求查询同一个 key，只让一个请求回源，其他等待结果。

第四，预热。
大促前提前加载热点商品、库存、配置。

第五，限流和降级。
热点过高时，限制非核心请求，返回兜底数据。

第六，分桶和分片。
把单点写压力拆到多个桶，最后再聚合。

Go 项目里，请求合并常用 singleflight 思路。
它能防止缓存击穿时大量 goroutine 同时打数据库。

---

## 【文本块 40】Q：缓存击穿时 Go 里怎么做请求合并？

缓存击穿指的是某个热点 key 过期的一瞬间，大量请求同时发现缓存没有，然后一起打到数据库。

解决方式之一是 singleflight：

> 对同一个 key，同一时刻只允许一个 goroutine 回源，其他 goroutine 等待这个结果。

Go 生态里 `golang.org/x/sync/singleflight` 就是这个思路。

面试里可以这样答：

> 热点 key 回源时，我会用 singleflight 做请求合并，保证同一个 key 同一时间只有一个请求查数据库，其他请求共享结果，避免缓存击穿把 DB 打爆。

---

# 十四、秒杀系统设计

## 【文本块 41】Q：秒杀系统一般怎么设计？

秒杀系统的核心目标有三个：

第一，抗住瞬时流量。
第二，防止库存超卖。
第三，保护核心系统不被打穿。

典型链路是：

```text
用户请求
 -> CDN/静态化
 -> 网关限流/验证码/风控
 -> 秒杀资格校验
 -> Redis 预扣库存
 -> 写入 MQ
 -> 异步创建订单
 -> 落库
 -> 用户查询结果
```

关键点：

第一，流量拦截在前面。
不要让所有请求都打到后端。可以用验证码、风控、限流、排队、静态化。

第二，库存判断前移。
不要所有请求都打数据库查库存。可以把库存放 Redis，用 Lua 原子扣减。

第三，下单异步化。
抢到资格后先返回“排队中”或“处理中”，然后通过 MQ 异步创建订单。

第四，防止超卖。
Redis 扣库存要原子，数据库落库也要有最终校验，比如条件更新或唯一约束。

第五，防止重复下单。
同一用户同一商品只能抢一次，用幂等 key 或唯一索引兜底。

第六，结果查询。
用户抢购后不一定立刻创建订单，可以提供查询接口查看成功、失败、排队中。

面试里可以这样回答：

> 秒杀设计不要让流量直接打数据库。我的思路是前端静态化和网关限流先拦一层，Redis 用 Lua 原子预扣库存，成功后写 MQ，订单服务异步消费创建订单，消费端做幂等，数据库用唯一索引和库存条件更新兜底，最后通过结果查询接口让用户感知状态。

---

## 【文本块 42】秒杀 Go 版落地关注点

Go 里做秒杀服务时，可以特别强调这些点：

第一，入口层用 middleware 做限流。
比如基于 IP、用户、商品 ID 做限流。

第二，所有下游调用带 context timeout。
Redis、MQ、DB 调用都不能无限等待。

第三，Redis Lua 保证扣库存和记录用户购买状态原子。
不要先 GET 再 SET，这会有并发问题。

第四，MQ producer 要处理发送失败。
可以结合本地消息表，避免 Redis 扣了库存但消息没发出去。

第五，消费者用 worker pool 控制落库并发。
不要 MQ 来多少就无限 goroutine 消费，避免打爆 DB。

第六，消费端幂等。
用 order_id、user_id + sku_id 唯一索引防重复。

第七，监控必须完善。
要看 QPS、限流数、Redis RT、MQ 堆积、订单创建成功率、失败率、DB RT。

---

# 十五、分库分表

## 【文本块 43】Q：什么时候需要分库分表？

当单库单表已经无法承载数据量或并发压力时，才考虑分库分表。

常见触发点：

第一，单表数据量太大。
比如订单表上亿行，索引变大，查询变慢，DDL 变重。

第二，写入压力太高。
单库写 QPS 达到瓶颈，主库 CPU、IO、锁竞争都很高。

第三，存储容量不足。
单机磁盘容量、备份恢复时间、主从同步压力都开始变成问题。

第四，热点表影响整体稳定性。
比如订单表压力太大，拖慢同库其他业务表。

第五，单库连接数、事务、锁竞争成为瓶颈。

但面试里一定要补一句：

> 分库分表是后手，不是先手。很多性能问题应该先看 SQL、索引、缓存、读写分离、归档、冷热分离。只有单库单表容量或写入确实扛不住，才上分库分表。

---

## 【文本块 44】Q：分库分表会带来哪些问题？

分库分表能解决容量和写入瓶颈，但会带来一堆复杂度。

第一，非分片键查询变难。
如果按 user_id 分片，现在要按 order_no 查，就不知道该查哪个库哪张表。

第二，跨库 join 变难。
原来一个 SQL join 就能解决，现在数据分散在多个库里，需要业务层组装或冗余。

第三，分布式事务更复杂。
一次业务可能写多个库，本地事务不够用。

第四，分页排序复杂。
比如全局按创建时间排序，要从多个分片查数据再归并排序。

第五，扩容迁移复杂。
分片数变化后，数据要迁移，路由规则要兼容。

第六，全局唯一 ID 需要单独设计。
不能再依赖单表自增 ID。

第七，运维成本上升。
库表更多，监控、备份、恢复、DDL、数据修复都更复杂。

所以面试里可以说：

> 分库分表不是银弹。它把单机容量问题变成了路由、事务、查询、排序、迁移和运维复杂度问题。因此只有业务规模逼到一定程度，才值得引入。

---

# 十六、分片键

## 【文本块 45】Q：分片键应该怎么选？

分片键是决定一条数据落到哪个库、哪张表的字段。

好的分片键应该满足：

第一，高频查询会带上它。
否则大量查询都变成跨片扫描。

第二，数据分布均匀。
不能大量数据集中在某几个分片。

第三，写入压力均匀。
不能所有新数据都落到一个分片。

第四，业务语义稳定。
字段不能经常变。

第五，尽量避免热点。
比如某些大商户、大用户的数据不能把单个分片打爆。

常见分片键：

* user_id
* order_id
* merchant_id
* tenant_id
* shop_id

不同业务选择不同。

比如 C 端订单系统，经常按用户查订单，可以按 user_id 分片。
如果后台运营经常按商户查订单，可以考虑 merchant_id。
如果订单号本身带路由信息，也可以按 order_id 分片。

面试里可以这样回答：

> 分片键选择的核心是匹配主要查询路径，同时保证数据和流量均匀。选错分片键后，非分片键查询会非常痛苦，所以分库分表前必须先分析业务查询模式，而不是拍脑袋选 user_id。

---

## 【文本块 46】Q：如果按 user_id 分片，但要按 order_no 查询怎么办？

这是典型的非分片键查询。

常见方案：

第一，order_no 里编码路由信息。
比如订单号生成时带上 user_id hash 或 shard_id。根据 order_no 能反推出分片。

第二，建立索引表。
维护一张 order_no -> user_id 或 order_no -> shard_id 的映射表。查询时先查索引表，再路由到对应分片。

第三，冗余宽表。
为某些查询场景专门建一份冗余数据。

第四，借助搜索引擎。
比如订单后台复杂查询可以同步到 Elasticsearch。

第五，业务限制。
不支持任意维度实时查询，或者要求必须带上用户 ID、商户 ID、时间范围。

面试里可以说：

> 分库分表后最怕没有分片键的查询。工程里通常通过路由信息编码、索引表、冗余表、搜索引擎或业务约束解决。核心是避免每次查询都扫所有分片。

---

# 十七、分片算法

## 【文本块 47】Q：常见分片算法有哪些？

常见有三类。

第一，范围分片。
比如按时间范围、ID 范围分片。

优点是简单，范围查询友好。
缺点是容易热点，比如最新数据都写到最后一个分片。

第二，哈希取模。
比如：

```text
shard = user_id % N
```

优点是数据分布比较均匀，实现简单。
缺点是扩容困难，N 变了以后大量数据要迁移。

第三，一致性哈希。
把节点映射到哈希环上，key 也映射到环上，顺时针找到节点。节点变化时，只迁移相邻区间的一部分数据。

优点是扩容迁移量较小。
缺点是实现复杂，需要虚拟节点改善均匀性。

面试里可以这样回答：

> 范围分片适合范围查询，但容易热点；取模分片简单均匀，但扩容迁移成本高；一致性哈希扩容友好，但实现和运维复杂一些。具体选型要看查询模式、扩容频率和业务热点。

---

## 【代码块 9】Go 简单取模分片路由

```go
package main

import (
	"fmt"
	"hash/fnv"
)

func HashString(s string) uint32 {
	h := fnv.New32a()
	_, _ = h.Write([]byte(s))
	return h.Sum32()
}

func RouteByUserID(userID string, dbCount int, tableCount int) (int, int) {
	hash := HashString(userID)

	dbIndex := int(hash % uint32(dbCount))
	tableIndex := int((hash / uint32(dbCount)) % uint32(tableCount))

	return dbIndex, tableIndex
}

func main() {
	db, table := RouteByUserID("user_1001", 4, 16)

	fmt.Println("db:", db)
	fmt.Println("table:", table)
}
```

---

## 【文本块 48】代码解释

这段代码展示了最简单的分片路由思路：

1. 对 user_id 做 hash
2. 根据 hash 定位数据库
3. 根据 hash 定位表

真实项目里还要考虑：

* 分片规则版本
* 扩容迁移
* 历史数据兼容
* 分片配置中心
* 路由结果缓存
* 分片键校验
* SQL 改写
* 跨片查询限制
* 分片规则测试

面试里可以说：

> 简单取模适合初期固定分片数的场景，但一旦分片数量变化，会涉及大量数据迁移。所以生产里要么提前规划足够分片，要么使用一致性哈希、虚拟分片或中间件来降低扩容成本。

---

# 十八、分库分表后的事务与查询

## 【文本块 49】Q：分库分表后事务怎么处理？

优先原则是：

> 尽量让一次事务落在同一个分片内。

比如订单按 user_id 分片，那么同一用户的一次下单相关数据尽量放到同一个分片，使用本地事务解决。

如果确实跨分片，就要看业务一致性要求：

强一致场景：
考虑 TCC、事务消息、严格状态机。

最终一致场景：
本地事务 + MQ + 幂等消费 + 补偿任务。

可拆分场景：
把一个大事务拆成多个小事务，通过状态流转串起来。

工程里不建议轻易做跨库强事务，因为复杂度和性能成本都很高。

面试里可以说：

> 分库分表后，我会优先通过分片键设计避免跨片事务。如果无法避免，就根据业务一致性要求选择 TCC、Saga 或本地消息表最终一致，而不是默认上一个重型分布式事务框架。

---

## 【文本块 50】Q：分库分表后分页排序怎么做？

如果查询只命中单个分片，分页排序和普通 MySQL 类似。

麻烦的是跨分片分页排序。

比如要查询所有订单按创建时间倒序分页：

```sql
SELECT * FROM orders ORDER BY created_at DESC LIMIT 20 OFFSET 1000;
```

如果有 16 个分片，就不能简单地每个分片查 20 条再拼起来。

常见做法：

第一，限制查询条件。
要求必须带分片键或时间范围，避免全局深分页。

第二，多分片归并排序。
每个分片查一部分，然后在应用层做 merge sort。

第三，使用搜索引擎。
把复杂查询同步到 Elasticsearch。

第四，使用游标分页。
避免 offset 深分页，使用 last_id、last_created_at 这类游标。

第五，建设宽表或报表库。
面向后台查询单独建数据模型。

面试里可以这样回答：

> 分库分表后跨片分页排序很复杂，尤其是深分页。工程里通常通过业务限制、游标分页、搜索引擎、宽表或离线报表解决，而不是让在线主库承接任意维度的全局分页查询。

---

# 十九、Go 项目里的分库分表落地

## 【文本块 51】Go 里怎么做分库分表？

Go 项目里常见有三种方式。

第一，业务代码自己路由。
根据分片键计算 db/table，然后选择对应 *sql.DB 或 gorm.DB。

优点是可控、轻量。
缺点是侵入业务，复杂 SQL 难处理。

第二，使用分库分表中间件。
例如数据库代理或中间层帮你做 SQL 路由。

优点是对业务侵入小。
缺点是中间件本身复杂，SQL 支持有限，排查链路变长。

第三，业务层做数据模型拆分。
比如订单主表按用户分片，后台查询走 ES 或报表库，不强求所有查询都走分片库。

Go 中比较常见的做法是轻量封装 Router：

```text
ShardRouter.Route(shardKey) -> DB + tableName
```

然后 repository 层根据路由结果拼接表名或选择连接。

---

## 【代码块 10】Go 分片 Router 示例

```go
package main

import (
	"database/sql"
	"fmt"
	"hash/fnv"
)

type Shard struct {
	DB        *sql.DB
	DBIndex   int
	TableBase string
	TableNum  int
}

type Router struct {
	shards []Shard
}

func hashKey(key string) uint32 {
	h := fnv.New32a()
	_, _ = h.Write([]byte(key))
	return h.Sum32()
}

func (r *Router) Route(key string) (*sql.DB, string, error) {
	if len(r.shards) == 0 {
		return nil, "", fmt.Errorf("no shard configured")
	}

	h := hashKey(key)

	shardIndex := int(h % uint32(len(r.shards)))
	shard := r.shards[shardIndex]

	tableIndex := int((h / uint32(len(r.shards))) % uint32(shard.TableNum))
	tableName := fmt.Sprintf("%s_%02d", shard.TableBase, tableIndex)

	return shard.DB, tableName, nil
}

func main() {
	router := &Router{
		shards: []Shard{
			{DBIndex: 0, TableBase: "orders", TableNum: 16},
			{DBIndex: 1, TableBase: "orders", TableNum: 16},
		},
	}

	_, table, err := router.Route("user_1001")
	fmt.Println(table, err)
}
```

---

## 【文本块 52】代码解释

这个示例展示的是分片路由的基本结构。

业务传入分片键，比如 user_id。
Router 算出库和表。
Repository 使用对应 DB 和 tableName 查询。

真实项目里要注意：

* 表名不能直接拼用户输入，必须由路由器生成
* 分片键不能为空
* SQL 要避免注入
* 分片规则要有版本
* 扩容迁移时要兼容旧规则
* 跨片查询要明确限制
* 每个分片连接池要单独监控
* 慢查询要能定位到具体分片

---

# 二十、服务优雅停机

## 【文本块 53】Q：Go 微服务为什么要做优雅停机？

在 Kubernetes 或普通部署环境中，服务发布、扩容、缩容、重启都很常见。

如果进程收到 SIGTERM 后立刻退出，可能导致：

* 正在处理的 HTTP 请求被中断
* 正在执行的 DB 事务被打断
* MQ 消费处理到一半
* 连接还没关闭
* 日志和指标没 flush
* 下游看到大量连接重置

优雅停机的目标是：

1. 停止接收新请求
2. 等待正在处理的请求完成
3. 超过一定时间则强制退出
4. 关闭 DB、Redis、MQ、日志等资源

Go 标准库 http.Server 提供了 Shutdown 方法，可以配合 context 使用。

---

## 【代码块 11】Go HTTP 优雅停机

```go
package main

import (
	"context"
	"log"
	"net/http"
	"os"
	"os/signal"
	"syscall"
	"time"
)

func main() {
	mux := http.NewServeMux()

	mux.HandleFunc("/hello", func(w http.ResponseWriter, r *http.Request) {
		time.Sleep(100 * time.Millisecond)
		_, _ = w.Write([]byte("hello"))
	})

	srv := &http.Server{
		Addr:    ":8080",
		Handler: mux,
	}

	go func() {
		log.Println("server started")
		if err := srv.ListenAndServe(); err != nil && err != http.ErrServerClosed {
			log.Fatal(err)
		}
	}()

	quit := make(chan os.Signal, 1)
	signal.Notify(quit, syscall.SIGINT, syscall.SIGTERM)

	<-quit
	log.Println("shutting down server...")

	ctx, cancel := context.WithTimeout(context.Background(), 5*time.Second)
	defer cancel()

	if err := srv.Shutdown(ctx); err != nil {
		log.Fatal("server forced to shutdown:", err)
	}

	log.Println("server exited")
}
```

---

## 【文本块 54】代码解释

`ListenAndServe` 在单独 goroutine 中启动服务。
主 goroutine 监听 SIGINT 和 SIGTERM。
收到信号后调用 `srv.Shutdown(ctx)`。

Shutdown 会停止接受新连接，并等待已有请求处理完成。
如果超过 5 秒还没结束，就根据 context 超时退出。

真实微服务还要处理：

* 从注册中心摘除实例
* readiness probe 先失败
* 等待负载均衡不再转发流量
* 停止 MQ 消费
* 等待 worker pool 完成
* flush 日志和指标
* 关闭数据库连接池

面试里可以说：

> Go 服务上线到 K8s 后，优雅停机非常重要。不能 SIGTERM 一来就退出，而应该先摘流量，再停止接新请求，等待正在处理的请求和消费任务完成，最后关闭资源。

---

# 二十一、项目化回答模板

## 【文本块 55】Q：如果面试官问你“设计一个高并发下单系统”，怎么答？

可以按这个结构：

第一，明确业务约束。
下单是读多还是写多？峰值 QPS 多大？是否有秒杀？库存是否强一致？是否允许异步？是否需要立即返回订单号？

第二，入口层抗流量。
网关限流、用户维度限流、风控、验证码、静态化、黑名单。

第三，服务层无状态扩容。
Go 服务多实例部署，handler 保持无状态，通过负载均衡分发流量。

第四，缓存层减轻读压力。
商品、价格、库存展示使用 Redis 或本地缓存，热点数据提前预热。

第五，库存扣减设计。
普通下单可以走数据库条件更新；秒杀场景可以 Redis Lua 预扣库存，后续异步落库。

第六，订单创建。
核心订单数据使用本地事务写入；如果异步化，则通过 MQ 削峰，消费者幂等创建订单。

第七，一致性保证。
本地事务 + outbox，消息重试，消费幂等，失败补偿，定时对账。

第八，数据库扩展。
先做 SQL 和索引优化、读写分离、冷热归档；如果订单表持续增长，再按 user_id 或 order_id 分库分表。

第九，稳定性治理。
超时、重试、熔断、降级、隔离、限流。重试必须幂等。

第十，可观测性。
日志、指标、trace，重点看 QPS、RT、错误率、Redis、DB、MQ 堆积、下游耗时。

这样回答会比直接说“Redis + MQ + 分库分表”成熟很多。

---

# 二十二、本章速记版

## 【文本块 56】分布式理论速记

分布式系统的难点来自网络不可靠、节点不可靠、数据同步有延迟。

CAP 中 P 基本不可避免。网络分区发生时，只能在 C 和 A 之间取舍。

CP 偏一致性，可能牺牲可用性。
AP 偏可用性，接受短暂不一致。

BASE 是互联网系统的工程化取舍：基本可用、软状态、最终一致。

最终一致不是“不管一致”，而是通过消息、重试、幂等、补偿、对账保证最后一致。

---

## 【文本块 57】微服务速记

微服务不是服务越小越好，而是按业务能力拆分。

拆分依据：

* 业务边界
* 团队边界
* 独立扩容
* 独立发布
* 故障隔离

不推荐按 Controller、Service、DAO 技术层拆。

服务太大是小单体，服务太小会导致调用链长、事务复杂、排查困难。

Go 微服务里 context 很关键，负责超时、取消、链路信息透传。

middleware/interceptor 是治理入口，可以做日志、recover、鉴权、限流、trace、metrics。

---

## 【文本块 58】服务治理速记

微服务真正难的是稳定调用。

服务治理包括：

* 注册发现
* 负载均衡
* 超时
* 重试
* 熔断
* 限流
* 降级
* 隔离
* 灰度
* 配置
* 健康检查
* 监控
* 日志
* trace

重试必须幂等。
超时必须设置。
熔断防止失败扩散。
降级保证核心功能。
限流保护系统入口和下游依赖。

---

## 【文本块 59】高并发速记

高并发设计不要一上来堆技术。

先看：

* 读多还是写多
* 峰值 QPS
* 是否允许异步
* 一致性要求
* 热点是否集中
* 瓶颈在哪里

优化分层：

* 流量层：CDN、限流、验证码、静态化
* 应用层：无状态扩容、批量化、池化、并行化
* 缓存层：本地缓存、Redis、多级缓存、热点隔离
* 存储层：索引、SQL、读写分离、分库分表
* 异步层：MQ 削峰填谷
* 稳定性层：熔断、降级、限流、监控

秒杀核心：

* 前面拦流量
* Redis 预扣库存
* MQ 异步下单
* 幂等防重复
* 数据库兜底防超卖
* 查询接口看结果

---

## 【文本块 60】分库分表速记

分库分表是为了解决单库单表容量和写入压力瓶颈。

触发点：

* 单表数据太大
* 索引太大
* 写入压力太高
* 单机资源瓶颈
* 备份恢复困难

分片键要满足：

* 高频查询带上它
* 分布均匀
* 避免热点
* 业务稳定

常见分片算法：

* 范围分片
* 哈希取模
* 一致性哈希

分库分表问题：

* 非分片键查询
* 跨库 join
* 分布式事务
* 分页排序
* 扩容迁移
* 全局 ID
* 运维复杂度

工程原则：

> 能单片解决就不要跨片，能本地事务解决就不要分布式事务，能通过索引和缓存解决就不要过早分库分表。

---

# 二十三、本章最终面试回答模板

## 【文本块 61】综合回答模板

如果面试官让我整体讲分布式、微服务、高并发和分库分表，我会这样回答：

分布式系统的核心问题不是“怎么调用另一个服务”，而是在网络不可靠、节点会故障、数据同步有延迟的前提下，如何在一致性、可用性和性能之间做取舍。CAP 说明了网络分区下 C 和 A 的取舍，BASE 则是互联网系统里更常见的工程实践，通过基本可用、软状态和最终一致来换取更高吞吐和可用性。

微服务本质上是按业务能力拆分系统，让每个服务可以独立开发、部署、扩容和演进。但微服务不是越小越好，也不是所有系统一开始都要拆。早期业务简单时单体效率更高，只有团队规模、业务复杂度、独立扩容和独立发布需求上来后，微服务才有收益。拆分时应该按业务边界拆，比如用户、订单、库存、支付，而不是按 Controller、Service、DAO 这种技术层拆。

Go 做微服务时，context 是请求生命周期的主线，要一路传到 DB、Redis、RPC、MQ 等下游调用，用于超时、取消和链路信息透传。middleware 和 interceptor 是治理入口，可以统一做日志、recover、鉴权、限流、metrics 和 trace。服务之间同步调用一般用 HTTP 或 gRPC，异步解耦和削峰填谷用 MQ。重试必须建立在幂等前提上，下游慢时要有超时、熔断、降级和隔离。

高并发设计不能一上来就套 Redis、MQ、分库分表，而是要先看业务约束：读多写多、峰值 QPS、一致性要求、是否允许异步、热点在哪里、瓶颈在哪一层。读多先考虑缓存和多级缓存，写峰值高考虑 MQ 削峰，热点 key 要做拆分、预热、请求合并和限流，数据库慢要先看 SQL、索引、连接池和读写分离，只有单库单表容量或写入压力真的顶不住时，才考虑分库分表。

分库分表主要解决单库单表数据量和并发写入瓶颈，但会带来非分片键查询、跨库 join、分布式事务、分页排序、扩容迁移、全局 ID 等复杂度。分片键要根据高频查询路径选择，同时保证数据和流量均匀。分片算法常见有范围分片、哈希取模和一致性哈希。分库分表后应尽量让一次事务落在同一个分片，跨片一致性通过 TCC、Saga、本地消息表、MQ 幂等消费、补偿任务和对账任务解决。

一句话总结：分布式系统设计的能力，不是背技术名词，而是能根据业务约束判断瓶颈，再在一致性、可用性、性能和复杂度之间做出合理取舍。Go 在这类系统中的优势，是用简单的同步代码风格、轻量 goroutine、context、middleware/interceptor 和清晰接口，把高并发网络服务和微服务治理做得足够直接、可控、可观测。
